# Synthetic Eval Workbench

Run a fresh synthetic-persona benchmark into `outputs/`, then inspect summary metrics, benchmark integrity, persona errors, and item-level bias tables. The notebook reads the supported minimal config surface from `.env`.

In [23]:
from pathlib import Path
import os
import sys

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

repo_root = Path.cwd()
while not (repo_root / "app").exists() and repo_root.parent != repo_root:
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

load_dotenv(repo_root / ".env", override=True)

from app.notebook_eval import (
    build_item_error_table,
    build_persona_error_table,
    load_benchmark_integrity,
    load_eval_metrics,
    load_eval_records,
    run_eval_notebook,
    split_item_error_table,
)


In [ ]:
# Primary run controls. Backend/model and benchmark defaults come from .env.
personas = 3
seed = 43
max_api_calls = 500
save_diagnostics = False
debug_outputs = True
trace_level = "off"
output_dir = repo_root / "outputs"
run_eval_now = True


In [25]:
detector_backend = os.getenv("DETECTOR_BACKEND", "openrouter").strip().lower() or "openrouter"
if detector_backend == "ollama":
    detector_target = os.getenv("OLLAMA_DETECTOR_MODEL", "qwen3.5:4b")
else:
    detector_target = os.getenv("OPENROUTER_DETECTOR_MODEL", "openrouter/auto")

print(f"Configured detector backend: {detector_backend} [{detector_target}]")
print("Persona runtime: deterministic simulator")


Configured detector backend: ollama [qwen3.5:4b]
Persona runtime: deterministic simulator


In [26]:
if run_eval_now:
    run_summary = run_eval_notebook(
        persona_count=personas,
        seed=seed,
        save_diagnostics=save_diagnostics,
        max_api_calls=max_api_calls,
        trace_level=trace_level,
        debug_outputs=debug_outputs,
        output_dir=output_dir,
    )
    output_dir = Path(run_summary["output_dir"])
else:
    run_summary = {"output_dir": str(Path(output_dir).resolve())}
    output_dir = Path(run_summary["output_dir"])

print(f"Artifacts: {output_dir}")


Running synthetic eval: personas=3, live_status=on
Stop policy: MIN_TURNS=20 | MAX_TURNS=40 | STOP_CONFIDENCE=0.66
Confidence model: CONF_SUPPORT_TAU=0.65 | CONF_DEPTH_WEIGHT=0.80 | CONF_COVERAGE_WEIGHT=0.30 | CONF_UP_ALPHA=0.65 | CONF_DECAY_STREAK_START=6 | CONF_DECAY_PER_TURN=0.002 | CONF_DECAY_MAX=0.01 | CONF_MAX_DROP_PER_TURN=0.01
Risk/extractor controls: RISK_SENTINEL_FLAG_THRESHOLD=0.15 | RISK_SENTINEL_SHORTCIRCUIT_THRESHOLD=1.1 | RISK_SENTINEL_ACTIVE_SHORTCIRCUIT=0 | EXTRACTOR_MIN_RECORDS_TARGET=1
[eval 1/3 persona=1] cycle=25 turn=23 stage=detector_graph conf=62.4% calls=54/500
[eval 2/3 persona=2] cycle=25 turn=23 stage=detector_graph conf=57.1% calls=122/500
[eval 3/3 persona=3] cycle=25 turn=23 stage=detector_graph conf=52.1% calls=179/500
item_f1=0.7698 objective=0.6798
Artifacts: /home/mdel2424/dev/eRisk_Honours/outputs


In [27]:
metrics = load_eval_metrics(output_dir)
benchmark_integrity = load_benchmark_integrity(output_dir)
records_df = load_eval_records(output_dir)
persona_error_df = build_persona_error_table(records_df)
item_error_df = build_item_error_table(records_df)
item_views = split_item_error_table(item_error_df)


In [28]:
def style_table(df: pd.DataFrame):
    if df.empty:
        return df
    format_map = {
        col: "{:.3f}"
        for col in ["avg_pred", "avg_true", "mean_error", "abs_mean_error", "bdi_error", "bdi_abs_error"]
        if col in df.columns
    }
    if "mean_error" in df.columns:
        return df.style.format(format_map).background_gradient(subset=["mean_error"], cmap="RdYlGn", vmin=-1.5, vmax=1.5)
    return df.style.format(format_map)


In [29]:
summary_df = pd.DataFrame([
    {
        "evaluation_mode": metrics.get("evaluation_mode", "synthetic"),
        "persona_count": metrics.get("persona_count", 0),
        "item_f1_macro_at_1": metrics.get("item_f1_macro_at_1", 0.0),
        "item_mae": metrics.get("item_mae", 0.0),
        "bdi_mae": metrics.get("bdi_mae", 0.0),
        "avg_turns_to_decision": metrics.get("avg_turns_to_decision", 0.0),
        "objective": metrics.get("objective", 0.0),
    }
])
display(summary_df)

integrity_df = pd.DataFrame([
    {
        "integrity_pass": benchmark_integrity.get("pass", False),
        "detector_backend": benchmark_integrity.get("detector", {}).get("backend", ""),
        "detector_target": benchmark_integrity.get("detector", {}).get("target", ""),
        "prior_manifest_exists": benchmark_integrity.get("prior_manifest", {}).get("exists", False),
        "prior_manifest_matches_current": benchmark_integrity.get("prior_manifest", {}).get("matches_current", None),
        "results_alignment_pass": benchmark_integrity.get("results_alignment", {}).get("pass", False),
        "manifest_consistency_pass": benchmark_integrity.get("manifest_consistency", {}).get("pass", False),
    }
])
display(integrity_df)

if benchmark_integrity.get("issues"):
    print("Integrity issues:", benchmark_integrity["issues"])

guardrail_mismatch_df = records_df.loc[
    ~records_df["finalizer_guardrail_consistency_ok"],
    [
        "persona_id",
        "family",
        "bdi_true",
        "bdi_pred",
        "finalizer_low_signal_guardrail_active",
        "finalizer_severe_recovery_mode_active",
        "finalizer_guardrail_bypass_source",
    ],
].reset_index(drop=True)
print(f"Guardrail invariant failures: {len(guardrail_mismatch_df)}")
if not guardrail_mismatch_df.empty:
    display(guardrail_mismatch_df)


,evaluation_mode,persona_count,item_f1_macro_at_1,item_mae,bdi_mae,avg_turns_to_decision,objective
0,synthetic,3,0.7698,0.6825,4.3333,24.0,0.6798


,integrity_pass,detector_backend,detector_target,prior_manifest_exists,prior_manifest_matches_current,results_alignment_pass,manifest_consistency_pass
0,True,ollama,qwen3.5:4b,True,False,True,True


Guardrail invariant failures: 0


In [30]:
display(style_table(persona_error_df.head(20)))


,persona_id,split,family,source,bdi_true,bdi_pred,bdi_error,bdi_abs_error
0,2,eval,somatic_evasive,synthetic,32,22,-10.000,10.000
1,1,eval,risk_leaning,synthetic,28,31,3.000,3.000
2,3,eval,cognitive_ruminative,synthetic,21,21,0.000,0.000


In [31]:
display(style_table(item_views["all_items"].sort_values(["mean_error", "item_id"]).reset_index(drop=True)))


,item_id,symptom_name,avg_pred,avg_true,mean_error,abs_mean_error,n_profiles
0,15,Loss of Energy,0.333,1.667,-1.333,1.333,3
1,19,Concentration Difficulty,0.333,1.333,-1.000,1.000,3
2,7,Self-Dislike,1.333,2.000,-0.667,0.667,3
3,8,Self-Criticalness,1.000,1.667,-0.667,0.667,3
4,9,Suicidal Thoughts or Wishes,0.000,0.667,-0.667,0.667,3
5,14,Worthlessness,1.333,2.000,-0.667,0.667,3
6,1,Sadness,0.667,1.000,-0.333,0.333,3
7,2,Pessimism,1.000,1.333,-0.333,0.333,3
8,12,Loss of Interest,0.667,1.000,-0.333,0.333,3
9,13,Indecisiveness,0.667,1.000,-0.333,0.333,3


In [32]:
display(style_table(item_views["under_predicted"].reset_index(drop=True)))


,item_id,symptom_name,avg_pred,avg_true,mean_error,abs_mean_error,n_profiles
0,15,Loss of Energy,0.333,1.667,-1.333,1.333,3
1,19,Concentration Difficulty,0.333,1.333,-1.000,1.000,3
2,7,Self-Dislike,1.333,2.000,-0.667,0.667,3
3,8,Self-Criticalness,1.000,1.667,-0.667,0.667,3
4,9,Suicidal Thoughts or Wishes,0.000,0.667,-0.667,0.667,3
5,14,Worthlessness,1.333,2.000,-0.667,0.667,3
6,1,Sadness,0.667,1.000,-0.333,0.333,3
7,2,Pessimism,1.000,1.333,-0.333,0.333,3
8,12,Loss of Interest,0.667,1.000,-0.333,0.333,3
9,13,Indecisiveness,0.667,1.000,-0.333,0.333,3


In [33]:
display(style_table(item_views["over_predicted"].reset_index(drop=True)))


,item_id,symptom_name,avg_pred,avg_true,mean_error,abs_mean_error,n_profiles
0,20,Tiredness or Fatigue,2.000,1.000,1.000,1.000,3
1,10,Crying,1.333,0.667,0.667,0.667,3
2,17,Irritability,1.333,0.667,0.667,0.667,3
3,3,Past Failure,1.333,1.000,0.333,0.333,3
4,5,Guilty Feelings,2.000,1.667,0.333,0.333,3
5,16,Changes in Sleeping Pattern,2.000,1.667,0.333,0.333,3
6,18,Changes in Appetite,1.333,1.000,0.333,0.333,3
7,21,Loss of Interest in Sex,1.667,1.333,0.333,0.333,3
